# BTL Học máy (608807) - Lớp 69CS1
## Mô hình Ensemble Dạng Tăng cường - HistGradientBoosting
**Người thực hiện:** Dương Hữu Hinh | MSSV: 0367869

---
### Giới thiệu mô hình
HistGradientBoostingRegressor sử dụng kỹ thuật rời rạc hóa đặc trưng thành 256 bins theo cơ chế tương tự LightGBM, tối ưu hóa tốc độ huấn luyện và khả năng nắm bắt phi tuyến phức tạp trong dữ liệu.

**Quy trình thực thi:**
1. Nạp tập train/test sạch qua `src.data.load_split()` (chống rò rỉ dữ liệu).
2. Khởi tạo Pipeline 5 bước với `TransformedTargetRegressor` (tối ưu RMSLE).
3. Đánh giá kiểm định chéo 5-Fold Cross Validation trên tập train.
4. Huấn luyện mô hình và đánh giá khách quan trên tập test độc lập.
5. Xuất sơ đồ đánh giá vào `output/plots/DanhGia_HistGB.png`.
6. Xuất file dự đoán đã format giá tiền vào `output/GiaNha_HistGB.csv`.


### 1. Thiết lập môi trường và nạp thư viện dùng chung

In [ ]:
import sys
from pathlib import Path

# Thiết lập đường dẫn thư mục gốc 03_SourceCode/
ROOT_DIR = Path.cwd().parents[1] if Path.cwd().name == 'train_models' else Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold, cross_validate
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from src.data import load_split
from src.pipeline import build_pipeline, get_models, RANDOM_STATE

print(f"Thư mục làm việc dự án: {ROOT_DIR}")


### 2. Nạp dữ liệu và tiền xử lý ngoại lai (Chỉ xử lý trên tập Train)

In [ ]:
# Nạp tập dữ liệu train/test đồng bộ theo thỏa thuận chung của nhóm
X_train, X_test, y_train, y_test = load_split()

# Lọc 3 mẫu ngoại lai có diện tích GrLivArea > 4000 chỉ trên tập train
keep = X_train["GrLivArea"] <= 4000
X_train, y_train = X_train[keep], y_train[keep]

print(f"[✓] Kích thước tập huấn luyện (X_train): {X_train.shape}")
print(f"[✓] Kích thước tập kiểm tra    (X_test):  {X_test.shape}")


### 3. Xây dựng Pipeline 5 bước cho mô hình HistGB

In [ ]:
model_name = "HistGB"
base_model = get_models()[model_name]
pipeline = build_pipeline(base_model, ordinal=True)
print(f"Cấu trúc pipeline cho mô hình {model_name}:")
pipeline


### 4. Đánh giá 5-Fold Cross Validation trên tập huấn luyện

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    'rmse': 'neg_root_mean_squared_error',
    'mae': 'neg_mean_absolute_error',
    'r2': 'r2',
    'rmsle': 'neg_root_mean_squared_log_error',
}

cv_res = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=scoring)

cv_rmsle = -cv_res['test_rmsle'].mean()
cv_r2 = cv_res['test_r2'].mean()
cv_rmse = -cv_res['test_rmse'].mean()
cv_mae = -cv_res['test_mae'].mean()

print("=" * 50)
print(f"KẾT QUẢ 5-FOLD CROSS VALIDATION (HistGB):")
print("=" * 50)
print(f"- RMSLE trung bình: {cv_rmsle:.4f} (+/- {cv_res['test_rmsle'].std():.4f})")
print(f"- R² Score:         {cv_r2:.4f} (+/- {cv_res['test_r2'].std():.4f})")
print(f"- RMSE trung bình:   ${cv_rmse:,.2f}")
print(f"- MAE trung bình:    ${cv_mae:,.2f}")


### 5. Huấn luyện toàn bộ tập train và đánh giá trên tập test độc lập

In [ ]:
# Huấn luyện lại trên toàn bộ tập train
pipeline.fit(X_train, y_train)

# Dự đoán trên tập kiểm tra (Test Set)
y_pred = pipeline.predict(X_test)

# Tính các chỉ số thực tế trên tập test
test_rmse = np.sqrt(mean_squared_error(y_test.values, y_pred))
test_mae = mean_absolute_error(y_test.values, y_pred)
test_r2 = r2_score(y_test.values, y_pred)
test_rmsle = np.sqrt(mean_squared_error(np.log1p(y_test.values), np.log1p(y_pred)))

print("=" * 50)
print(f"HIỆU NĂNG TRÊN TẬP TEST ĐỘC LẬP (HistGB):")
print("=" * 50)
print(f"- R² Score: {test_r2:.4f}")
print(f"- RMSLE:    {test_rmsle:.4f}")
print(f"- RMSE:     ${test_rmse:,.2f}")
print(f"- MAE:      ${test_mae:,.2f}")


### 6. Sơ đồ đánh giá trực quan mô hình HistGB

In [ ]:
residuals = y_test.values - y_pred
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle(f"SƠ ĐỒ ĐÁNH GIÁ MÔ HÌNH: HISTGB", fontsize=16, fontweight='bold', y=0.98)

# 1. Actual vs Predicted
ax1 = axes[0, 0]
ax1.scatter(y_test.values, y_pred, alpha=0.6, color='#1f77b4', edgecolor='k', linewidth=0.5, label='Mẫu test')
min_val = min(y_test.values.min(), y_pred.min()) * 0.9
max_val = max(y_test.values.max(), y_pred.max()) * 1.05
ax1.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Đường chuẩn lý tưởng (y=x)')
ax1.set_xlabel('Giá thực tế (USD)', fontsize=11, fontweight='bold')
ax1.set_ylabel('Giá dự đoán (USD)', fontsize=11, fontweight='bold')
ax1.set_title(f'Giá thực tế vs Giá dự đoán\nR² = {test_r2:.4f} | RMSLE = {test_rmsle:.4f}', fontsize=12)
ax1.legend(loc='upper left')
ax1.grid(True, linestyle=':', alpha=0.6)
ax1.xaxis.set_major_formatter('${x:,.0f}')
ax1.yaxis.set_major_formatter('${x:,.0f}')

# 2. Residuals Plot
ax2 = axes[0, 1]
ax2.scatter(y_pred, residuals, alpha=0.6, color='#2ca02c', edgecolor='k', linewidth=0.5)
ax2.axhline(0, color='red', linestyle='--', linewidth=2)
ax2.set_xlabel('Giá dự đoán (USD)', fontsize=11, fontweight='bold')
ax2.set_ylabel('Phần dư (Y_thực tế - Y_dự đoán) [USD]', fontsize=11, fontweight='bold')
ax2.set_title('Biểu đồ phân tán phần dư (Residuals vs Predicted)', fontsize=12)
ax2.grid(True, linestyle=':', alpha=0.6)
ax2.xaxis.set_major_formatter('${x:,.0f}')
ax2.yaxis.set_major_formatter('${x:,.0f}')

# 3. Residual Distribution
ax3 = axes[1, 0]
ax3.hist(residuals, bins=30, color='#ff7f0e', edgecolor='black', alpha=0.7, density=True)
mu, std = np.mean(residuals), np.std(residuals)
x_norm = np.linspace(min(residuals), max(residuals), 100)
p_norm = 1 / (std * np.sqrt(2 * np.pi)) * np.exp(-(x_norm - mu) ** 2 / (2 * std ** 2))
ax3.plot(x_norm, p_norm, 'k-', linewidth=2, label=f'Chuẩn (μ={mu:,.0f}, σ={std:,.0f})')
ax3.set_xlabel('Phần dư (USD)', fontsize=11, fontweight='bold')
ax3.set_ylabel('Mật độ xác suất', fontsize=11, fontweight='bold')
ax3.set_title('Phân phối sai số phần dư', fontsize=12)
ax3.legend(loc='upper right', fontsize=9)
ax3.grid(True, linestyle=':', alpha=0.6)
ax3.xaxis.set_major_formatter('${x:,.0f}')

# 4. Metrics Card
ax4 = axes[1, 1]
ax4.axis('off')
summary_data = [
    ['Thuật toán / Mô hình', 'HistGB'],
    ['Hệ số xác định (R² Score)', f'{test_r2:.4f}'],
    ['Độ đo hàm mục tiêu (RMSLE)', f'{test_rmsle:.4f}'],
    ['Sai số bình phương trung bình (RMSE)', f'${test_rmse:,.2f}'],
    ['Sai số tuyệt đối trung bình (MAE)', f'${test_mae:,.2f}'],
    ['Sai số lớn nhất (Max Error)', f'${np.max(np.abs(residuals)):,.2f}'],
    ['Số mẫu tập kiểm tra', f'{len(y_test):,d} căn'],
]
table = ax4.table(cellText=summary_data, colWidths=[0.55, 0.45], loc='center', cellLoc='left')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 1.8)
for (r_i, c_i), cell in table.get_celld().items():
    cell.set_edgecolor('#cccccc')
    if c_i == 0:
        cell.set_text_props(weight='bold', color='#1b365d')
        cell.set_facecolor('#f0f4f8')
    if r_i == 0:
        cell.set_facecolor('#1b365d')
        cell.set_text_props(weight='bold', color='#ffffff')
ax4.set_title('TỔNG KẾT ĐỘ ĐO ĐÁNH GIÁ ĐỊNH LƯỢNG', fontsize=12, pad=15)

plt.tight_layout()
plot_file = ROOT_DIR / 'output' / 'plots' / f'DanhGia_HistGB.png'
plot_file.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(plot_file, dpi=200, bbox_inches='tight')
plt.show()
print(f'[✓] Đã lưu sơ đồ đánh giá vào: {plot_file}')


### 7. Xuất file kết quả dự đoán đã format tiền tệ vào `output/GiaNha_HistGB.csv`

In [ ]:
test_ids = pd.read_csv(ROOT_DIR / 'data' / 'processed' / 'test_ids.csv')['Id'].values
abs_diff = np.abs(y_test.values - y_pred)
pct_diff = (abs_diff / y_test.values) * 100

output_df = pd.DataFrame({
    'Id': test_ids,
    'GiaNha_ThucTe': np.round(y_test.values, 2),
    'GiaNha_DuDoan': np.round(y_pred, 2),
    'GiaNha_DinhDang': [f"${val:,.2f}" for val in y_pred],
    'ChenhLech_USD': np.round(abs_diff, 2),
    'TiLe_SaiLech_%': [f"{p:.2f}%" for p in pct_diff]
})

output_path = ROOT_DIR / 'output' / f'GiaNha_HistGB.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)
output_df.to_csv(output_path, index=False)
print(f'[✓] Đã xuất file kết quả: {output_path.name} ({len(output_df)} dòng)')
output_df.head(10)
